# Buildings mangrove attribution using area-distance weighting

This notebook implements the patch-level attribution formula discussed for buildings only.

For each building with avoided EAD, it:
1. Finds all mangrove patches whose 5000 m buffer intersects the building geometry.
2. Computes the original building-to-mangrove distance for each matched pair.
3. Computes the raw weight as `mangrove_area / distance`.
4. Normalizes weights within each building.
5. Attributes the building's avoided EAD across its matched mangrove patches.

If a building has one or more zero-distance matches, attribution is split across those zero-distance mangroves by area only.

In [ ]:
from pathlib import Path
import pathlib
import sys

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas
from IPython.display import display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

project_root = pathlib.Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
robyn_libraries_path = project_root / 'robyns_libraries'
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

In [ ]:
# User parameters
SCENARIO = 'minimum'  # 'minimum' or 'maximum'
BUFFER_M = 5000.0
ATTRIBUTE_ONLY_POSITIVE_AVOIDED = True
USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER = True
ZERO_DISTANCE_TOLERANCE_M = 0.001
MAP_DISPLAY_QUANTILE = 0.995

if SCENARIO not in {'minimum', 'maximum'}:
    raise ValueError("SCENARIO must be 'minimum' or 'maximum'.")
if BUFFER_M <= 0:
    raise ValueError('BUFFER_M must be > 0.')
if ZERO_DISTANCE_TOLERANCE_M < 0:
    raise ValueError('ZERO_DISTANCE_TOLERANCE_M must be >= 0.')
if not (0 < MAP_DISPLAY_QUANTILE <= 1):
    raise ValueError('MAP_DISPLAY_QUANTILE must be within (0, 1].')

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{SCENARIO}_scenario'
damage_estimates_path = results_path / 'damage_estimates'
asset_ead_csv = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
building_geometry_gpkg = damage_estimates_path / 'buildings_assigned_economic_activity_areas_asset_damages_groupedby.gpkg'
mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
out_dir = damage_estimates_path / 'mangrove_attribution_area_distance_buildings'
out_dir.mkdir(parents=True, exist_ok=True)

for required_path in [asset_ead_csv, building_geometry_gpkg, mangrove_path, jamaica_boundary_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required path: {required_path}')

method_label_base = f'area_distance_{int(round(BUFFER_M))}m'
method_label = f'{method_label_base}_nn_fallback' if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER else method_label_base
print(f'SCENARIO: {SCENARIO}')
print(f'BUFFER_M: {BUFFER_M:,.0f}')
print(f'ATTRIBUTE_ONLY_POSITIVE_AVOIDED: {ATTRIBUTE_ONLY_POSITIVE_AVOIDED}')
print(f'USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER: {USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER}')
print(f'Output folder: {out_dir}')

In [ ]:
# Load buildings avoided-EAD table
asset_ead = pandas.read_csv(asset_ead_csv)

building_ead = asset_ead.loc[
    (asset_ead['Asset'] == 'buildings_assigned_economic_activity')
    & (asset_ead['Layer'] == 'areas'),
    [
        'Sector',
        'Subsector',
        'Asset',
        'Layer',
        'Asset_ID',
        'EAD_With_Mangroves_USD',
        'EAD_Without_Mangroves_USD',
        'Avoided_EAD_USD',
        'Avoided_EAD_Share_of_Baseline',
    ],
].copy()

if building_ead.empty:
    raise ValueError('No building rows found in the asset-level EAD table.')

if ATTRIBUTE_ONLY_POSITIVE_AVOIDED:
    building_ead = building_ead.loc[building_ead['Avoided_EAD_USD'] > 0].copy()

building_ead['Asset_ID'] = building_ead['Asset_ID'].astype(str)
building_ead = building_ead.drop_duplicates(subset=['Asset_ID']).reset_index(drop=True)

print(f'Building rows used: {len(building_ead):,}')
print(f'Total avoided EAD used (USD): {float(building_ead["Avoided_EAD_USD"].sum()):,.2f}')
display(building_ead.head(10))

In [ ]:
# Load one geometry per building
building_geometries = geopandas.read_file(building_geometry_gpkg)
if building_geometries.crs is None:
    raise ValueError('Building geometry CRS is missing.')
if str(building_geometries.crs).upper() != 'EPSG:3448':
    building_geometries = building_geometries.to_crs('EPSG:3448')
if 'osm_id' not in building_geometries.columns:
    raise KeyError("Expected 'osm_id' in the building geometry file.")

building_geometries['Asset_ID'] = building_geometries['osm_id'].astype(str)
building_geometries = building_geometries.merge(building_ead, on='Asset_ID', how='inner')

if building_geometries['Asset_ID'].duplicated().any():
    duplicate_count = int(building_geometries['Asset_ID'].duplicated().sum())
    raise ValueError(f'Expected one geometry per building Asset_ID, found {duplicate_count:,} duplicates.')

asset_key_columns = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
building_columns = asset_key_columns + [
    'EAD_With_Mangroves_USD',
    'EAD_Without_Mangroves_USD',
    'Avoided_EAD_USD',
    'Avoided_EAD_Share_of_Baseline',
    'geometry',
]
building_gdf = geopandas.GeoDataFrame(building_geometries[building_columns].copy(), geometry='geometry', crs='EPSG:3448')

print(f'Building geometries loaded: {len(building_gdf):,}')
display(building_gdf.head(5))

In [ ]:
# Load mangroves and compute patch areas
mangroves = geopandas.read_file(mangrove_path)
if mangroves.crs is None:
    raise ValueError('Mangrove CRS is missing.')
if str(mangroves.crs).upper() != 'EPSG:3448':
    mangroves = mangroves.to_crs('EPSG:3448')

if 'ID' in mangroves.columns:
    mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
else:
    mangroves = mangroves.reset_index(drop=True)
    mangroves['Mangrove_ID'] = numpy.arange(1, len(mangroves) + 1)

mangroves['Mangrove_Area_m2'] = mangroves.geometry.area
mangroves['Mangrove_Area_ha'] = mangroves['Mangrove_Area_m2'] / 10000.0

mangrove_base_columns = ['Mangrove_ID', 'Mangrove_Area_m2', 'Mangrove_Area_ha']
for mangrove_attribute_column in ['Parish', 'HECTARES', 'TYPE']:
    if mangrove_attribute_column in mangroves.columns:
        mangrove_base_columns.append(mangrove_attribute_column)

print(f'Mangrove patches loaded: {len(mangroves):,}')
print(f'Total mangrove area (ha): {float(mangroves["Mangrove_Area_ha"].sum()):,.2f}')
display(mangroves[mangrove_base_columns].head(10))

In [ ]:
# Candidate building-mangrove pairs from the 5000 m mangrove buffer

def apply_area_distance_weights(pair_df):
    if pair_df.empty:
        pair_df = pair_df.copy()
        pair_df['is_zero_distance'] = pandas.Series(dtype=bool)
        pair_df['has_zero_distance_match'] = pandas.Series(dtype=bool)
        pair_df['weight_raw'] = pandas.Series(dtype=float)
        pair_df['weight_sum'] = pandas.Series(dtype=float)
        pair_df['weight'] = pandas.Series(dtype=float)
        pair_df['Avoided_EAD_USD_attributed'] = pandas.Series(dtype=float)
        return pair_df

    pair_df = pair_df.copy()
    pair_df['is_zero_distance'] = pair_df['distance_m'] <= ZERO_DISTANCE_TOLERANCE_M
    pair_df['has_zero_distance_match'] = pair_df.groupby(asset_key_columns)['is_zero_distance'].transform('max').astype(bool)
    pair_df['weight_raw'] = 0.0

    zero_distance_rows = pair_df['has_zero_distance_match'] & pair_df['is_zero_distance']
    positive_distance_rows = (~pair_df['has_zero_distance_match']) & (pair_df['distance_m'] > ZERO_DISTANCE_TOLERANCE_M)

    pair_df.loc[zero_distance_rows, 'weight_raw'] = pair_df.loc[zero_distance_rows, 'Mangrove_Area_m2']
    pair_df.loc[positive_distance_rows, 'weight_raw'] = (
        pair_df.loc[positive_distance_rows, 'Mangrove_Area_m2']
        / pair_df.loc[positive_distance_rows, 'distance_m']
    )

    pair_df['weight_sum'] = pair_df.groupby(asset_key_columns)['weight_raw'].transform('sum')
    if (pair_df['weight_sum'] <= 0).any():
        bad_assets = pair_df.loc[pair_df['weight_sum'] <= 0, asset_key_columns].drop_duplicates()
        raise ValueError(f'Found assets with non-positive weight sums: {len(bad_assets):,}')

    pair_df['weight'] = pair_df['weight_raw'] / pair_df['weight_sum']
    pair_df['Avoided_EAD_USD_attributed'] = pair_df['Avoided_EAD_USD'] * pair_df['weight']
    return pair_df

mangrove_buffers = mangroves[mangrove_base_columns + ['geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(BUFFER_M)

candidate_pairs = geopandas.sjoin(
    building_gdf,
    mangrove_buffers[['Mangrove_ID', 'geometry']],
    how='left',
    predicate='intersects',
)

candidate_pairs['nearby_mangrove_count'] = candidate_pairs.groupby(asset_key_columns)['Mangrove_ID'].transform(
    lambda matched_mangrove_ids: matched_mangrove_ids.notna().sum()
)
candidate_pairs['nearby_mangrove_count'] = candidate_pairs['nearby_mangrove_count'].fillna(0).astype(int)

mangrove_lookup = pandas.DataFrame(mangroves[mangrove_base_columns].copy())
mangrove_lookup['mangrove_geometry'] = mangroves.geometry.values

buffer_pairs = candidate_pairs.dropna(subset=['Mangrove_ID']).copy()
buffer_pairs['Mangrove_ID'] = buffer_pairs['Mangrove_ID'].astype(int)
buffer_pairs = buffer_pairs.merge(mangrove_lookup, on='Mangrove_ID', how='left')

if not buffer_pairs.empty:
    mangrove_geometry_series = geopandas.GeoSeries(buffer_pairs['mangrove_geometry'], index=buffer_pairs.index, crs='EPSG:3448')
    buffer_pairs['distance_m'] = buffer_pairs.geometry.distance(mangrove_geometry_series)
    buffer_pairs = apply_area_distance_weights(buffer_pairs)
    buffer_pairs['Attribution_Source'] = 'buffer_area_distance'
    buffer_pairs['nearest_tie_count'] = 0
else:
    buffer_pairs['distance_m'] = pandas.Series(dtype=float)
    buffer_pairs = apply_area_distance_weights(buffer_pairs)
    buffer_pairs['Attribution_Source'] = pandas.Series(dtype=str)
    buffer_pairs['nearest_tie_count'] = pandas.Series(dtype=int)

buffer_matched_asset_ids = set(buffer_pairs['Asset_ID'].astype(str).unique())
unmatched_buildings = building_gdf.loc[~building_gdf['Asset_ID'].isin(buffer_matched_asset_ids)].copy()

fallback_pairs = pandas.DataFrame()
if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER and not unmatched_buildings.empty:
    fallback_pairs = geopandas.sjoin_nearest(
        unmatched_buildings,
        mangroves[mangrove_base_columns + ['geometry']],
        how='left',
        distance_col='distance_m',
    )
    fallback_pairs = fallback_pairs.dropna(subset=['Mangrove_ID']).copy()
    fallback_pairs['Mangrove_ID'] = fallback_pairs['Mangrove_ID'].astype(int)
    fallback_pairs['nearby_mangrove_count'] = 0
    fallback_pairs['nearest_tie_count'] = fallback_pairs.groupby(asset_key_columns)['Mangrove_ID'].transform('size').astype(int)
    fallback_pairs = apply_area_distance_weights(fallback_pairs)
    fallback_pairs['Attribution_Source'] = 'nearest_outside_buffer'

combined_pair_columns = [
    'Sector',
    'Subsector',
    'Asset',
    'Layer',
    'Asset_ID',
    'EAD_With_Mangroves_USD',
    'EAD_Without_Mangroves_USD',
    'Avoided_EAD_USD',
    'Avoided_EAD_Share_of_Baseline',
    'geometry',
    'Mangrove_ID',
    'Mangrove_Area_m2',
    'Mangrove_Area_ha',
    'distance_m',
    'nearby_mangrove_count',
    'nearest_tie_count',
    'is_zero_distance',
    'has_zero_distance_match',
    'weight_raw',
    'weight_sum',
    'weight',
    'Avoided_EAD_USD_attributed',
    'Attribution_Source',
]

matched_pairs = pandas.concat(
    [
        buffer_pairs[combined_pair_columns],
        fallback_pairs[combined_pair_columns] if len(fallback_pairs) > 0 else pandas.DataFrame(columns=combined_pair_columns),
    ],
    ignore_index=True,
)
matched_pairs = geopandas.GeoDataFrame(matched_pairs, geometry='geometry', crs='EPSG:3448')

buffer_matched_building_count = int(buffer_pairs['Asset_ID'].nunique())
fallback_building_count = int(fallback_pairs['Asset_ID'].nunique()) if len(fallback_pairs) > 0 else 0
all_attributed_building_count = int(matched_pairs['Asset_ID'].nunique()) if len(matched_pairs) > 0 else 0

print(f'Buildings with at least one mangrove inside the buffer: {buffer_matched_building_count:,}')
print(f'Buildings outside the buffer: {len(unmatched_buildings):,}')
print(f'Buildings using nearest-neighbour fallback: {fallback_building_count:,}')
print(f'Total attributed buildings after combining both methods: {all_attributed_building_count:,}')
print(f'Combined building-mangrove rows: {len(matched_pairs):,}')
print(f'Buildings with zero-distance matches: {matched_pairs.loc[matched_pairs["has_zero_distance_match"], "Asset_ID"].nunique() if len(matched_pairs) > 0 else 0:,}')
display(
    matched_pairs[[
        'Asset_ID',
        'Mangrove_ID',
        'Attribution_Source',
        'Mangrove_Area_ha',
        'distance_m',
        'nearby_mangrove_count',
        'nearest_tie_count',
        'weight_raw',
        'weight',
        'Avoided_EAD_USD',
        'Avoided_EAD_USD_attributed',
    ]].head(10)
)

In [ ]:
# QA checks and building-level attribution status
weight_check = (
    matched_pairs.groupby(asset_key_columns, as_index=False)['weight']
    .sum()
    .rename(columns={'weight': 'weight_sum_check'})
)

building_attribution = (
    matched_pairs.groupby(asset_key_columns, as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Avoided_EAD_USD_attributed': 'Attributed_EAD_USD'})
)

buffer_status = buffer_pairs[asset_key_columns].drop_duplicates().copy()
buffer_status['Has_Buffer_Mangrove'] = 1

fallback_status = fallback_pairs[asset_key_columns].drop_duplicates().copy() if len(fallback_pairs) > 0 else pandas.DataFrame(columns=asset_key_columns)
fallback_status['Used_Nearest_Fallback'] = 1

building_status = building_gdf[asset_key_columns + ['Avoided_EAD_USD', 'geometry']].copy()
building_status = building_status.merge(weight_check, on=asset_key_columns, how='left')
building_status = building_status.merge(building_attribution, on=asset_key_columns, how='left')
building_status = building_status.merge(buffer_status, on=asset_key_columns, how='left')
building_status = building_status.merge(fallback_status, on=asset_key_columns, how='left')
building_status['weight_sum_check'] = building_status['weight_sum_check'].fillna(0.0)
building_status['Attributed_EAD_USD'] = building_status['Attributed_EAD_USD'].fillna(0.0)
building_status['Has_Buffer_Mangrove'] = building_status['Has_Buffer_Mangrove'].fillna(0).astype(int)
building_status['Used_Nearest_Fallback'] = building_status['Used_Nearest_Fallback'].fillna(0).astype(int)
building_status['Has_Attributed_Mangrove'] = (building_status['Attributed_EAD_USD'] > 0).astype(int)
building_status['Unattributed_EAD_USD'] = building_status['Avoided_EAD_USD'] - building_status['Attributed_EAD_USD']

max_weight_error = float((building_status.loc[building_status['Has_Attributed_Mangrove'] == 1, 'weight_sum_check'] - 1.0).abs().max()) if (building_status['Has_Attributed_Mangrove'] == 1).any() else 0.0
input_total_usd = float(building_status['Avoided_EAD_USD'].sum())
attributed_total_usd = float(building_status['Attributed_EAD_USD'].sum())
unattributed_total_usd = float(building_status['Unattributed_EAD_USD'].sum())
matched_building_count = int(building_status['Has_Attributed_Mangrove'].sum())

print(f'Total input avoided EAD (USD): {input_total_usd:,.2f}')
print(f'Total attributed avoided EAD (USD): {attributed_total_usd:,.2f}')
print(f'Total unattributed avoided EAD (USD): {unattributed_total_usd:,.2f}')
print(f'Buildings matched within buffer: {int(building_status["Has_Buffer_Mangrove"].sum()):,}')
print(f'Buildings using nearest fallback: {int(building_status["Used_Nearest_Fallback"].sum()):,}')
print(f'Total attributed buildings: {matched_building_count:,}/{len(building_status):,}')
print(f'Max weight-sum error across attributed buildings: {max_weight_error:.12f}')

if not numpy.isclose(input_total_usd, attributed_total_usd + unattributed_total_usd, atol=1e-6):
    raise ValueError('Input total does not equal attributed plus unattributed totals.')

In [ ]:
# Mangrove summaries and saved outputs
asset_to_mangrove_output = matched_pairs[[
    'Sector',
    'Subsector',
    'Asset',
    'Layer',
    'Asset_ID',
    'Avoided_EAD_USD',
    'Mangrove_ID',
    'Mangrove_Area_m2',
    'Mangrove_Area_ha',
    'distance_m',
    'nearby_mangrove_count',
    'nearest_tie_count',
    'weight_raw',
    'weight',
    'Avoided_EAD_USD_attributed',
    'Attribution_Source',
]].copy()

mangrove_total_summary = (
    asset_to_mangrove_output.groupby('Mangrove_ID', as_index=False)
    .agg(
        Total_Avoided_EAD_USD_attributed=('Avoided_EAD_USD_attributed', 'sum'),
        Building_Count=('Asset_ID', 'nunique'),
        Mean_Building_Distance_m=('distance_m', 'mean'),
        Buffer_Attributed_Building_Count=('Attribution_Source', lambda values: int((pandas.Series(values) == 'buffer_area_distance').sum())),
        Fallback_Attributed_Building_Count=('Attribution_Source', lambda values: int((pandas.Series(values) == 'nearest_outside_buffer').sum())),
    )
    .sort_values('Total_Avoided_EAD_USD_attributed', ascending=False)
    .reset_index(drop=True)
)

mangrove_attribution_map = mangroves[mangrove_base_columns + ['geometry']].copy()
for mangrove_attribute_column in ['Parish', 'HECTARES', 'TYPE']:
    if mangrove_attribute_column in mangroves.columns and mangrove_attribute_column not in mangrove_attribution_map.columns:
        mangrove_attribution_map[mangrove_attribute_column] = mangroves[mangrove_attribute_column]

mangrove_attribution_map = mangrove_attribution_map.merge(mangrove_total_summary, on='Mangrove_ID', how='left')
for summary_column in [
    'Total_Avoided_EAD_USD_attributed',
    'Building_Count',
    'Buffer_Attributed_Building_Count',
    'Fallback_Attributed_Building_Count',
]:
    mangrove_attribution_map[summary_column] = mangrove_attribution_map[summary_column].fillna(0.0)

run_summary = pandas.DataFrame([
    {
        'Scenario': SCENARIO,
        'Buffer_m': BUFFER_M,
        'Attribute_Only_Positive_Avoided': ATTRIBUTE_ONLY_POSITIVE_AVOIDED,
        'Use_Nearest_Mangrove_Fallback_For_Outside_Buffer': USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER,
        'Input_Building_Count': int(len(building_status)),
        'Buffer_Matched_Building_Count': int(building_status['Has_Buffer_Mangrove'].sum()),
        'Fallback_Building_Count': int(building_status['Used_Nearest_Fallback'].sum()),
        'Attributed_Building_Count': matched_building_count,
        'Input_Avoided_EAD_USD': input_total_usd,
        'Attributed_Total_USD': attributed_total_usd,
        'Unattributed_Total_USD': unattributed_total_usd,
        'Fraction_Attributed_pct': (100.0 * attributed_total_usd / input_total_usd) if abs(input_total_usd) > 0 else numpy.nan,
        'Zero_Distance_Building_Count': int(matched_pairs.loc[matched_pairs['has_zero_distance_match'], 'Asset_ID'].nunique()) if len(matched_pairs) > 0 else 0,
        'Mean_Buffer_Candidate_Mangrove_Count': float(asset_to_mangrove_output.loc[asset_to_mangrove_output['Attribution_Source'] == 'buffer_area_distance', ['Asset_ID', 'nearby_mangrove_count']].drop_duplicates()['nearby_mangrove_count'].mean()) if (asset_to_mangrove_output['Attribution_Source'] == 'buffer_area_distance').any() else numpy.nan,
        'Mean_Fallback_Distance_m': float(asset_to_mangrove_output.loc[asset_to_mangrove_output['Attribution_Source'] == 'nearest_outside_buffer', 'distance_m'].mean()) if (asset_to_mangrove_output['Attribution_Source'] == 'nearest_outside_buffer').any() else numpy.nan,
    }
])

asset_to_mangrove_csv = out_dir / f'buildings_asset_to_mangrove_attribution_{method_label}.csv'
building_status_csv = out_dir / f'buildings_asset_attribution_status_{method_label}.csv'
mangrove_total_csv = out_dir / f'mangrove_attribution_total_buildings_{method_label}.csv'
run_summary_csv = out_dir / f'run_summary_buildings_{method_label}.csv'
mangrove_map_gpkg = out_dir / f'mangrove_attribution_total_buildings_{method_label}.gpkg'

asset_to_mangrove_output.to_csv(asset_to_mangrove_csv, index=False)
building_status.drop(columns=['geometry']).to_csv(building_status_csv, index=False)
mangrove_total_summary.to_csv(mangrove_total_csv, index=False)
run_summary.to_csv(run_summary_csv, index=False)
mangrove_attribution_map.to_file(mangrove_map_gpkg, driver='GPKG')

print('Saved outputs:')
print(f' - {asset_to_mangrove_csv}')
print(f' - {building_status_csv}')
print(f' - {mangrove_total_csv}')
print(f' - {run_summary_csv}')
print(f' - {mangrove_map_gpkg}')

In [ ]:
# Save one row per building with patch count and mangrove IDs
if 'matched_pairs' not in globals() or 'building_status' not in globals():
    raise ValueError('Run the attribution and status cells first so matched_pairs and building_status exist.')

ordered_pairs = matched_pairs.copy()
ordered_pairs['Mangrove_ID_Int'] = ordered_pairs['Mangrove_ID'].astype(int)
ordered_pairs = ordered_pairs.sort_values(
    asset_key_columns + ['Avoided_EAD_USD_attributed', 'Mangrove_ID_Int'],
    ascending=[True, True, True, True, True, False, True],
).reset_index(drop=True)

ordered_pairs['mangrove_weight_label'] = ordered_pairs.apply(
    lambda row: f"{int(row['Mangrove_ID_Int'])} ({row['weight']:.6f})",
    axis=1,
)
ordered_pairs['mangrove_attributed_usd_label'] = ordered_pairs.apply(
    lambda row: f"{int(row['Mangrove_ID_Int'])} ({row['Avoided_EAD_USD_attributed']:.2f})",
    axis=1,
)

asset_patch_summary_core = (
    ordered_pairs.groupby(asset_key_columns, as_index=False)
    .agg(
        Mangrove_Patch_Count=('Mangrove_ID_Int', 'nunique'),
        Mean_Distance_m=('distance_m', 'mean'),
        Min_Distance_m=('distance_m', 'min'),
        Max_Distance_m=('distance_m', 'max'),
    )
)

mangrove_id_list = (
    ordered_pairs.groupby(asset_key_columns)['Mangrove_ID_Int']
    .apply(lambda values: ', '.join(map(str, values.tolist())))
    .rename('Mangrove_ID_List')
    .reset_index()
)

mangrove_weight_list = (
    ordered_pairs.groupby(asset_key_columns)['mangrove_weight_label']
    .apply(lambda values: '; '.join(values.tolist()))
    .rename('Mangrove_ID_Weight_List')
    .reset_index()
)

mangrove_attributed_usd_list = (
    ordered_pairs.groupby(asset_key_columns)['mangrove_attributed_usd_label']
    .apply(lambda values: '; '.join(values.tolist()))
    .rename('Mangrove_ID_Attributed_USD_List')
    .reset_index()
)

attribution_source_list = (
    ordered_pairs.groupby(asset_key_columns)['Attribution_Source']
    .apply(lambda values: ', '.join(sorted(set(values.tolist()))))
    .rename('Attribution_Source_List')
    .reset_index()
)

asset_patch_summary = building_status[[
    'Sector',
    'Subsector',
    'Asset',
    'Layer',
    'Asset_ID',
    'Avoided_EAD_USD',
    'Attributed_EAD_USD',
    'Unattributed_EAD_USD',
    'Has_Buffer_Mangrove',
    'Used_Nearest_Fallback',
    'Has_Attributed_Mangrove',
]].copy()

for summary_table in [
    asset_patch_summary_core,
    mangrove_id_list,
    mangrove_weight_list,
    mangrove_attributed_usd_list,
    attribution_source_list,
]:
    asset_patch_summary = asset_patch_summary.merge(summary_table, on=asset_key_columns, how='left')

asset_patch_summary['Mangrove_Patch_Count'] = asset_patch_summary['Mangrove_Patch_Count'].fillna(0).astype(int)
for string_column in [
    'Mangrove_ID_List',
    'Mangrove_ID_Weight_List',
    'Mangrove_ID_Attributed_USD_List',
    'Attribution_Source_List',
]:
    asset_patch_summary[string_column] = asset_patch_summary[string_column].fillna('')

asset_patch_summary = asset_patch_summary.sort_values(
    ['Mangrove_Patch_Count', 'Avoided_EAD_USD', 'Asset_ID'],
    ascending=[False, False, True],
).reset_index(drop=True)

asset_patch_summary_csv = out_dir / f'buildings_asset_patch_count_and_ids_{method_label}.csv'
asset_patch_summary.to_csv(asset_patch_summary_csv, index=False)

print(f'Saved: {asset_patch_summary_csv}')
print('Patch-count summary:')
display(asset_patch_summary['Mangrove_Patch_Count'].describe())
print('Top 10 buildings by number of attributed mangrove patches:')
display(
    asset_patch_summary[[
        'Asset_ID',
        'Avoided_EAD_USD',
        'Mangrove_Patch_Count',
        'Mangrove_ID_List',
        'Mangrove_ID_Attributed_USD_List',
    ]].head(10)
)

In [ ]:
# Review summary outputs
display(run_summary)
print('Top 20 mangrove patches by attributed avoided EAD (USD):')
display(mangrove_total_summary.head(20))

fig, axis = plt.subplots(figsize=(10, 6))
plot_data = mangrove_total_summary.head(15).sort_values('Total_Avoided_EAD_USD_attributed')
axis.barh(plot_data['Mangrove_ID'].astype(str), plot_data['Total_Avoided_EAD_USD_attributed'], color='#2f6f4f')
axis.set_xlabel('Attributed avoided EAD (USD)')
axis.set_ylabel('Mangrove_ID')
axis.set_title(f'Top 15 mangrove patches | buildings | {SCENARIO} scenario')
plt.tight_layout()
plt.show()

In [ ]:
# Map mangrove patches by attributed avoided EAD (USD)
if 'mangrove_attribution_map' not in globals():
    raise ValueError('Run the summary/output cell first so mangrove_attribution_map exists.')

jamaica_boundary = geopandas.read_file(jamaica_boundary_path)
if jamaica_boundary.crs is None:
    raise ValueError('Jamaica boundary CRS is missing.')
if str(jamaica_boundary.crs).upper() != 'EPSG:3448':
    jamaica_boundary = jamaica_boundary.to_crs('EPSG:3448')

mangrove_plot = mangrove_attribution_map.copy()
if mangrove_plot.crs is None:
    raise ValueError('Mangrove attribution map CRS is missing.')
if str(mangrove_plot.crs).upper() != 'EPSG:3448':
    mangrove_plot = mangrove_plot.to_crs('EPSG:3448')

value_column = 'Total_Avoided_EAD_USD_attributed'
if value_column not in mangrove_plot.columns:
    raise KeyError(f"Column '{value_column}' not found in mangrove attribution map.")

values = mangrove_plot[value_column].fillna(0.0)
positive_values = values.clip(lower=0.0)
display_cap = float(positive_values.quantile(MAP_DISPLAY_QUANTILE))
if display_cap <= 0:
    display_cap = float(positive_values.max()) if float(positive_values.max()) > 0 else 1.0

mangrove_plot['_plot_value'] = positive_values.clip(upper=display_cap)
positive_mangroves = mangrove_plot[mangrove_plot[value_column] > 0].copy()
non_positive_mangroves = mangrove_plot[mangrove_plot[value_column] <= 0].copy()

green_colormap = LinearSegmentedColormap.from_list(
    'white_to_dark_green',
    ['#eef8ef', '#b5dfb8', '#5fbf6a', '#238b45', '#005a32'],
    N=256,
)
value_norm = plt.Normalize(vmin=0.0, vmax=display_cap)

figure, axis = plt.subplots(figsize=(10.8, 9.4))
axis.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=axis, color='#9a9a9a', linewidth=0.5, zorder=1)

if not non_positive_mangroves.empty:
    non_positive_mangroves.plot(
        ax=axis,
        color='#d9d9d9',
        edgecolor='#9a9a9a',
        linewidth=0.3,
        alpha=0.95,
        zorder=2,
    )

if not positive_mangroves.empty:
    positive_mangroves.plot(
        ax=axis,
        column='_plot_value',
        cmap=green_colormap,
        norm=value_norm,
        edgecolor='#1f5a2e',
        linewidth=0.45,
        alpha=0.98,
        zorder=3,
    )

scalar_mappable = ScalarMappable(norm=value_norm, cmap=green_colormap)
scalar_mappable.set_array([])
colorbar = figure.colorbar(
    scalar_mappable,
    ax=axis,
    orientation='horizontal',
    fraction=0.045,
    pad=0.02,
)
colorbar.set_label('Attributed avoided EAD (USD)')
colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda tick_value, tick_position: f'{tick_value:,.0f}'))

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

fallback_title_suffix = ' + nearest fallback' if USE_NEAREST_MANGROVE_FALLBACK_FOR_OUTSIDE_BUFFER else ''
axis.set_title(
    f'Buildings: mangrove attributed avoided EADs ({SCENARIO} scenario, {int(round(BUFFER_M))} m buffer{fallback_title_suffix})',
    fontsize=12,
)
axis.set_axis_off()
plt.tight_layout()

map_png = out_dir / f'mangrove_attribution_map_buildings_{method_label}_q{int(MAP_DISPLAY_QUANTILE * 1000)}.png'
figure.savefig(map_png, dpi=300, bbox_inches='tight')
print(f'Saved: {map_png}')
plt.show()